In [5]:
import sys
import os

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))

sys.path.insert(0, str(project_root))
print(f"Project root added to sys.path: {project_root}")

Project root added to sys.path: /Volumes/DuceDrive/DataWork/churn-ml-project


In [6]:
import pandas as pd
import numpy as np
import joblib

from src.db_utils import get_engine
from src.model_config import FEATURE_CONFIG

engine = get_engine()
df = pd.read_sql("SELECT * FROM telco_churn", con=engine)

X = df[FEATURE_CONFIG.numeric_features + FEATURE_CONFIG.categorical_features].copy()
y = (df[FEATURE_CONFIG.target_col] == "Yes").astype(int)

In [7]:
from src.config import BASE_DIR

log_reg_model = joblib.load(BASE_DIR / "models" / "log_reg_pipeline.joblib")
rf_model = joblib.load(BASE_DIR / "models" / "random_forest_pipeline.joblib")

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

rf_pipeline: Pipeline = rf_model
preprocessor: ColumnTransformer = rf_pipeline.named_steps["preprocessor"]
rf_clf = rf_pipeline.named_steps["clf"]

# Get one-hot encoded feature names for categorical
ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]

cat_feature_names = ohe.get_feature_names_out(FEATURE_CONFIG.categorical_features)
all_feature_names = np.concatenate([FEATURE_CONFIG.numeric_features, cat_feature_names])

len(all_feature_names), all_feature_names[:10]

(51,
 array(['tenure', 'MonthlyCharges', 'TotalCharges', 'charge_ratio',
        'gender_Female', 'gender_Male', 'SeniorCitizen_0',
        'SeniorCitizen_1', 'Partner_No', 'Partner_Yes'], dtype=object))